# Surface Anomaly Screening Product

## Mueang Feuang, Laos

This notebook creates a reproducible Stage 1 product from multi-date Sentinel-2 Level-2A surface reflectance. It ranks exposed-surface spectral anomalies within the AOI. It does not detect gold, identify a mineral species, estimate grade, or replace geological and field validation. Scores are relative to this AOI and run.


In [ ]:
%pip -q install pystac-client planetary-computer stackstac rioxarray geopandas scipy matplotlib pandas numpy

In [ ]:
from pathlib import Path
import json
import platform
import sys
from datetime import datetime, timezone

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import planetary_computer
import rioxarray
import stackstac
import xarray as xr
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from pystac_client import Client
from rasterio.features import geometry_mask, shapes
from scipy.ndimage import binary_closing, binary_opening, label
from shapely.geometry import Polygon, shape

OUTPUT_DIR = Path("/Users/teerathap/Desktop/gold/surface_anomaly_product")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AOI_CRS = "EPSG:4326"
UTM_CRS = "EPSG:32648"
ANALYSIS_RESOLUTION_M = 20
MAX_SCENES = 6
MIN_CLEAR_OBSERVATIONS = 2
EXPOSED_NDVI_MAX = 0.35
MIN_TARGET_PIXELS = 8
SEARCH_DATES = "2023-01-01/2025-12-31"
np.random.seed(42)

In [ ]:
aoi_coordinates = [[102.049173,18.539069],[102.101129,18.539604],[102.101351,18.522001],[102.113321,18.522184],[102.113799,18.493660],[102.100343,18.493635],[102.100503,18.465557],[102.084232,18.465398],[102.084035,18.478125],[102.075567,18.478085],[102.075618,18.493525],[102.049337,18.493117],[102.049173,18.539069]]
aoi = Polygon(aoi_coordinates)
aoi_gdf = gpd.GeoDataFrame({"name":["Mueang Feuang AOI"]}, geometry=[aoi], crs=AOI_CRS)
aoi_utm = aoi_gdf.to_crs(UTM_CRS)
AOI_BBOX = aoi.bounds
AOI_AREA_KM2 = float(aoi_utm.geometry.area.iloc[0] / 1_000_000)
print(f"AOI area: {AOI_AREA_KM2:.3f} km²")

In [ ]:
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=planetary_computer.sign_inplace)
search = catalog.search(collections=["sentinel-2-l2a"], intersects=aoi.__geo_interface__, datetime=SEARCH_DATES, query={"eo:cloud_cover":{"lt":35}})
items = list(search.item_collection())
scene_table = pd.DataFrame([{
    "item_id":item.id,
    "datetime":item.datetime,
    "cloud_cover_pct":item.properties.get("eo:cloud_cover"),
    "tile":item.properties.get("s2:mgrs_tile"),
    "processing_baseline":item.properties.get("s2:processing_baseline")
} for item in items])
scene_table = scene_table.sort_values(["datetime","processing_baseline","cloud_cover_pct"], ascending=[True,False,True]).drop_duplicates("datetime", keep="first")
scene_table = scene_table.sort_values(["cloud_cover_pct","datetime"]).reset_index(drop=True)
assert not scene_table.empty, "No Sentinel-2 L2A scenes satisfy the search criteria."
selected_ids = scene_table.head(MAX_SCENES).item_id.tolist()
selected_items = [item for item in items if item.id in selected_ids]
selected_table = scene_table[scene_table.item_id.isin(selected_ids)].sort_values("datetime").reset_index(drop=True)
assert len(selected_items) >= MIN_CLEAR_OBSERVATIONS, "Too few scenes for a multi-date product."
display(selected_table)

In [ ]:
assets = ["B02","B03","B04","B08","B11","B12","SCL"]
stack = stackstac.stack(selected_items, assets=assets, bounds_latlon=AOI_BBOX, epsg=32648, resolution=ANALYSIS_RESOLUTION_M, dtype="float64", fill_value=np.nan, rescale=False).compute()
reflectance = stack.sel(band=["B02","B03","B04","B08","B11","B12"]) / 10000.0
scl = stack.sel(band="SCL")
blue, green, red, nir, swir1, swir2 = [reflectance.sel(band=b) for b in ["B02","B03","B04","B08","B11","B12"]]
ndvi = (nir-red)/(nir+red+1e-8)
ndwi = (green-nir)/(green+nir+1e-8)
clear_land = xr.apply_ufunc(np.isin, scl, np.array([4,5]))
exposed_surface = clear_land & (ndvi < EXPOSED_NDVI_MAX) & (ndwi < 0) & (swir1 > 0.02)
surface_support = exposed_surface.sum("time")
valid_surface = surface_support >= MIN_CLEAR_OBSERVATIONS
surface_reflectance = reflectance.where(exposed_surface).median("time", skipna=True)
print(f"Selected scenes: {len(selected_items)}")
print(f"Eligible exposed surface: {float(valid_surface.mean()*100):.2f}% of raster extent")

In [ ]:
def robust_unit(layer, mask):
    values = layer.where(mask)
    low = float(values.quantile(0.02, skipna=True))
    high = float(values.quantile(0.98, skipna=True))
    return ((layer-low)/max(high-low,1e-8)).clip(0,1), low, high

surface_blue, surface_green, surface_red, surface_nir, surface_swir1, surface_swir2 = [surface_reflectance.sel(band=b) for b in ["B02","B03","B04","B08","B11","B12"]]
ferric_proxy = surface_red/(surface_blue+1e-8)
hydroxyl_proxy = surface_swir1/(surface_swir2+1e-8)
brightness_proxy = (surface_blue+surface_green+surface_red+surface_nir+surface_swir1+surface_swir2)/6
ferric_score, ferric_p02, ferric_p98 = robust_unit(ferric_proxy, valid_surface)
hydroxyl_score, hydroxyl_p02, hydroxyl_p98 = robust_unit(hydroxyl_proxy, valid_surface)
brightness_score, brightness_p02, brightness_p98 = robust_unit(brightness_proxy, valid_surface)
relative_surface_evidence = (0.45*ferric_score + 0.40*hydroxyl_score + 0.15*brightness_score).where(valid_surface)
aoi_mask = geometry_mask(aoi_utm.geometry, out_shape=valid_surface.shape, transform=valid_surface.rio.transform(), invert=True)
analysis_mask = valid_surface.values & aoi_mask
score_mask = xr.DataArray(analysis_mask, coords=valid_surface.coords, dims=valid_surface.dims)
target_threshold = float(relative_surface_evidence.where(score_mask).quantile(0.95, skipna=True))
print(f"Relative evidence threshold: {target_threshold:.4f}")

In [ ]:
priority_raw = relative_surface_evidence.values >= target_threshold
priority_open = binary_opening(priority_raw & analysis_mask, structure=np.ones((3,3),dtype=bool))
priority_closed = binary_closing(priority_open, structure=np.ones((3,3),dtype=bool))
zone_labels, _ = label(priority_closed, structure=np.ones((3,3),dtype=np.int8))
zone_sizes = np.bincount(zone_labels.ravel())
priority_mask = np.isin(zone_labels, np.where(zone_sizes >= MIN_TARGET_PIXELS)[0]) & (zone_labels > 0) & analysis_mask
records = []
for zone_id in np.unique(zone_labels[priority_mask]):
    zone_mask = (zone_labels == zone_id) & priority_mask
    zone_shapes = list(shapes(zone_mask.astype(np.uint8), mask=zone_mask, transform=relative_surface_evidence.rio.transform()))
    if zone_shapes:
        records.append({
            "Zone_ID":int(zone_id),
            "Relative_Evidence":float(np.nanmean(relative_surface_evidence.values[zone_mask])),
            "Clear_Observations":float(np.nanmean(surface_support.values[zone_mask])),
            "geometry":shape(zone_shapes[0][0])
        })
assert records, "No connected target polygons remain after filtering."
targets = gpd.GeoDataFrame(records, crs=UTM_CRS)
targets["Area_ha"] = targets.geometry.area / 10000
targets = targets.sort_values(["Relative_Evidence","Area_ha"], ascending=[False,False]).reset_index(drop=True)
targets["Target_ID"] = [f"T{i:02d}" for i in range(1,len(targets)+1)]
targets["Final_Rank"] = np.arange(1,len(targets)+1)
print(f"Priority polygons: {len(targets)}")
print(f"Priority area: {targets.Area_ha.sum():.2f} ha")

In [ ]:
dem_items = list(catalog.search(collections=["cop-dem-glo-30"], intersects=aoi.__geo_interface__).item_collection())
assert dem_items, "No Copernicus DEM tiles found for this AOI."
dem = stackstac.stack([dem_items[0]], assets=["data"], bounds_latlon=AOI_BBOX, epsg=32648, resolution=30, dtype="float64", fill_value=np.nan, rescale=False).squeeze(["time","band"],drop=True).compute()
pixel_y = float(np.abs(np.diff(dem.y.values).mean()))
pixel_x = float(np.abs(np.diff(dem.x.values).mean()))
dz_dy, dz_dx = np.gradient(dem.values, pixel_y, pixel_x)
slope = xr.DataArray(np.degrees(np.arctan(np.hypot(dz_dx,dz_dy))), coords=dem.coords, dims=dem.dims)
terrain = []
for _, row in targets.iterrows():
    zone_mask = geometry_mask([row.geometry], out_shape=dem.shape, transform=dem.rio.transform(), invert=True)
    terrain.append({"Target_ID":row.Target_ID,"Elevation_Median_m":float(np.nanmedian(dem.values[zone_mask])),"Slope_Mean_deg":float(np.nanmean(slope.values[zone_mask])),"Slope_P90_deg":float(np.nanpercentile(slope.values[zone_mask],90))})
targets = targets.merge(pd.DataFrame(terrain), on="Target_ID", how="left")
targets["Terrain_Observation_Class"] = np.select([targets.Slope_P90_deg<=25,targets.Slope_P90_deg<=40],["Favourable","Moderate"],default="Steep")

In [ ]:
centroids_wgs84 = gpd.GeoSeries(targets.geometry.centroid, crs=UTM_CRS).to_crs(AOI_CRS)
targets["Longitude"] = centroids_wgs84.x
targets["Latitude"] = centroids_wgs84.y
columns = ["Target_ID","Final_Rank","Zone_ID","Area_ha","Relative_Evidence","Clear_Observations","Elevation_Median_m","Slope_Mean_deg","Slope_P90_deg","Terrain_Observation_Class","Longitude","Latitude","geometry"]
target_export = targets[columns].copy()
target_export.to_file(OUTPUT_DIR/"surface_anomaly_targets.geojson", driver="GeoJSON")
target_export.drop(columns="geometry").to_csv(OUTPUT_DIR/"surface_anomaly_targets.csv",index=False)
manifest = {
    "product":"Mueang Feuang surface anomaly screening",
    "generated_utc":datetime.now(timezone.utc).isoformat(),
    "purpose":"Relative ranking of exposed-surface Sentinel-2 spectral anomalies",
    "limitation":"Not a gold detection or mineral identification product",
    "aoi_area_km2":round(AOI_AREA_KM2,3),
    "analysis_crs":UTM_CRS,
    "analysis_grid_m":ANALYSIS_RESOLUTION_M,
    "sentinel_items":selected_table.assign(datetime=selected_table.datetime.astype(str)).to_dict("records"),
    "surface_rule":{"scl_classes":[4,5],"ndvi_lt":EXPOSED_NDVI_MAX,"ndwi_lt":0,"min_clear_observations":MIN_CLEAR_OBSERVATIONS},
    "proxy_formulas":{"ferric":"B04/B02","hydroxyl":"B11/B12","brightness":"mean(B02,B03,B04,B08,B11,B12)"},
    "weights":{"ferric":0.45,"hydroxyl":0.40,"brightness":0.15},
    "normalization":{"method":"AOI valid-surface percentile","lower":0.02,"upper":0.98,"priority_quantile":0.95},
    "minimum_target_area_ha":round(MIN_TARGET_PIXELS*ANALYSIS_RESOLUTION_M**2/10000,2),
    "terrain_dem":"Copernicus DEM GLO-30, analysed at native 30 m grid",
    "software":{"python":sys.version,"platform":platform.platform()}
}
(OUTPUT_DIR/"product_manifest.json").write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
display(target_export.drop(columns="geometry").round(4))

In [ ]:
rgb = reflectance.sel(band=["B04","B03","B02"]).median("time",skipna=True).transpose("y","x","band").values
lo, hi = np.nanpercentile(rgb,[2,98])
rgb_display = np.clip((rgb-lo)/max(hi-lo,1e-8),0,1)
extent = [float(reflectance.x.min()),float(reflectance.x.max()),float(reflectance.y.min()),float(reflectance.y.max())]
aoi_x, aoi_y = aoi_utm.geometry.iloc[0].exterior.xy
fig, axes = plt.subplots(1,3,figsize=(21,7),constrained_layout=True)
axes[0].imshow(rgb_display,extent=extent,origin="upper")
axes[0].plot(aoi_x,aoi_y,color="white",linewidth=2)
axes[0].set_title("Sentinel-2 Median True Color")
axes[1].imshow(analysis_mask,extent=extent,origin="upper",cmap=ListedColormap(["#17202a","#e8f6f3"]))
axes[1].plot(aoi_x,aoi_y,color="#ff3155",linewidth=2)
axes[1].set_title("Eligible Exposed Surface")
evidence_image=axes[2].imshow(relative_surface_evidence.values,extent=extent,origin="upper",cmap="magma",vmin=0,vmax=1)
targets.boundary.plot(ax=axes[2],color="#ffe66d",linewidth=1.5)
axes[2].plot(aoi_x,aoi_y,color="white",linewidth=1.4)
axes[2].set_title("Relative Surface Evidence")
fig.colorbar(evidence_image,ax=axes[2],label="AOI-relative evidence")
for ax in axes: ax.set_axis_off()
fig.savefig(OUTPUT_DIR/"screening_overview.png",dpi=300,bbox_inches="tight",facecolor="white")
plt.show()

In [ ]:
targets["Field_Priority"] = np.select([targets.Final_Rank<=3,targets.Final_Rank<=6],["Tier 1","Tier 2"],default="Tier 3")
colors={"Tier 1":"#ff3155","Tier 2":"#ff9f1c","Tier 3":"#ffe66d"}
fig,ax=plt.subplots(figsize=(12,12),constrained_layout=True)
ax.imshow(rgb_display,extent=extent,origin="upper")
aoi_utm.boundary.plot(ax=ax,color="white",linewidth=5)
aoi_utm.boundary.plot(ax=ax,color="black",linewidth=2.5)
for name,color in colors.items():
    subset = targets[targets.Field_Priority==name]
    if not subset.empty: subset.plot(ax=ax,facecolor=color,edgecolor="black",linewidth=.8,alpha=.75)
label_offsets = [(0,-16),(0,16),(-20,14),(-18,0),(22,-12),(0,16),(-20,-14),(18,14)]
for i,(_,row) in enumerate(targets.iterrows()):
    point=row.geometry.representative_point()
    ax.annotate(row.Target_ID,(point.x,point.y),xytext=label_offsets[i % len(label_offsets)],textcoords="offset points",ha="center",va="center",fontsize=8,fontweight="bold",bbox={"boxstyle":"round,pad=0.22","facecolor":"white","edgecolor":colors[row.Field_Priority]},arrowprops={"arrowstyle":"-","color":"white","linewidth":0.8})
ax.set_title("Surface Anomaly Screening Map\nMueang Feuang, Laos",fontsize=16,fontweight="bold")
ax.legend(handles=[Patch(facecolor=color,edgecolor="black",label=name) for name,color in colors.items()],title="Provisional field priority",loc="lower left")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.grid(alpha=.25,color="white")
fig.savefig(OUTPUT_DIR/"surface_anomaly_screening_map.png",dpi=300,bbox_inches="tight",facecolor="white")
plt.show()

## Product outputs

- `surface_anomaly_targets.geojson`: target polygons in UTM 48N with WGS84 centroids
- `surface_anomaly_targets.csv`: field-ready target schedule
- `product_manifest.json`: source scene IDs, thresholds, formulae, and provenance
- `screening_overview.png`: QA view for masking and evidence
- `surface_anomaly_screening_map.png`: provisional field-priority map

## Interpretation rules

- High evidence means unusual relative to valid exposed surfaces inside this AOI.
- It does not identify gold or a specific mineral.
- Field work must compare targets against geology, structure, access, artisanal workings, and ground observations.
- Hyperspectral mineral identification is a separate calibrated stage.

## Method sources

- https://sentinels.copernicus.eu/sentinel-data-access/sentinel-products/sentinel-2-data-products/collection-1-level-2a
- https://sentinels.copernicus.eu/documents/d/sentinel/s2-pdgs-cs-di-psd-v15-0
- https://sentinels.copernicus.eu/documents/247904/685211/Sentinel-2_User_Handbook
- https://doi.org/10.1016/j.oregeorev.2022.105154
- https://documentation.dataspace.copernicus.eu/Data/Others/CCM.html
